In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin
import time

# Website categories
categories = {
    "Laptops": "https://webscraper.io/test-sites/e-commerce/allinone/computers/laptops",
    "Tablets": "https://webscraper.io/test-sites/e-commerce/allinone/computers/tablets",
    "Phones": "https://webscraper.io/test-sites/e-commerce/allinone/phones/touch"
}

# User-Agent
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/140.0.0.0 Safari/537.36"
}

products = []

# Scrape each category
for category, url in categories.items():

    print(f"\nScraping: {category}")

    page_url = url

    while page_url:

        response = requests.get(page_url, headers=headers)

        print("Status Code:", response.status_code)

        if response.status_code != 200:
            print("Page could not be loaded.")
            break

        soup = BeautifulSoup(response.text, "html.parser")

        # Find products
        for product in soup.select(".thumbnail"):

            name_tag = product.select_one(".title")
            price_tag = product.select_one(".price")
            description_tag = product.select_one(".description")
            review_tag = product.select_one(".ratings .pull-right")
            link_tag = product.select_one(".title")

            name = name_tag.get("title", "").strip() if name_tag else ""
            price = price_tag.get_text(strip=True) if price_tag else ""
            description = description_tag.get_text(strip=True) if description_tag else ""
            reviews = review_tag.get_text(strip=True) if review_tag else ""

            product_url = ""

            if link_tag and link_tag.get("href"):
                product_url = urljoin(page_url, link_tag["href"])

            products.append({
                "Category": category,
                "Product Name": name,
                "Price": price,
                "Description": description,
                "Reviews": reviews,
                "Product URL": product_url
            })

        # Find next page
        next_button = soup.select_one("ul.pagination li a")

        next_url = None

        for link in soup.select("ul.pagination li a"):
            if link.get_text(strip=True).lower() == "next":
                next_url = urljoin(page_url, link.get("href"))
                break

        page_url = next_url

        time.sleep(1)


# Convert to DataFrame
df = pd.DataFrame(products)

# Remove duplicate products
df.drop_duplicates(subset=["Product Name", "Category"], inplace=True)

# Save dataset
df.to_csv("ecommerce_products.csv", index=False)

print("\n--------------------------------")
print("Scraping Completed!")
print("Total Products:", len(df))
print("--------------------------------")

print(df.head(10))


Scraping: Laptops
Status Code: 200

Scraping: Tablets
Status Code: 200

Scraping: Phones
Status Code: 200

--------------------------------
Scraping Completed!
Total Products: 114
--------------------------------
   Category                            Product Name    Price  \
0   Laptops              Asus VivoBook X441NA-GA190  $295.99   
1   Laptops      Prestigio SmartBook 133S Dark Grey     $299   
2   Laptops           Prestigio SmartBook 133S Gold     $299   
3   Laptops                           Aspire E1-510  $306.99   
4   Laptops                       Lenovo V110-15IAP  $321.94   
6   Laptops  Hewlett Packard 250 G6 Dark Ash Silver  $364.46   
7   Laptops             Acer Aspire 3 A315-31 Black   $372.7   
8   Laptops                Acer Aspire A315-31-C33J  $379.94   
9   Laptops               Acer Aspire ES1-572 Black  $379.95   
11  Laptops                   Acer Aspire 3 A315-21  $393.88   

                                          Description Reviews  \
0   Asus VivoBoo

In [ ]:
print(df.shape)
print(df.columns)
print(df.info())
print(df.isnull().sum())


(114, 6)
Index(['Category', 'Product Name', 'Price', 'Description', 'Reviews',
       'Product URL'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
Index: 114 entries, 0 to 144
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Category      114 non-null    object
 1   Product Name  114 non-null    object
 2   Price         114 non-null    object
 3   Description   114 non-null    object
 4   Reviews       114 non-null    object
 5   Product URL   114 non-null    object
dtypes: object(6)
memory usage: 6.2+ KB
None
Category        0
Product Name    0
Price           0
Description     0
Reviews         0
Product URL     0
dtype: int64
